# Notebook 07 — Robustness Checks: do the findings survive?

Notebooks 05 and 06 report the results of the thesis. This notebook asks the questions an examiner will ask about them, and answers each with the data. It changes nothing upstream and re-estimates the models of Notebooks 05–06 exactly as they were specified.

| Part | Question | How it is tested |
|---|---|---|
| **1 · Multiple testing** | Notebooks 05–06 test many coefficients. How many of the "significant" ones would we expect by chance? | Benjamini–Hochberg false-discovery-rate correction over all plot-position and timing tests, and separately over all community tests. |
| **2 · Foresight or imitation?** | Films whose stories later films came to resemble ("foresight") do better. Is that because they were ahead of their time, or because **hits get copied**? | (a) Foresight measured only against films released 1–2 years later — too soon to be copies, since a film takes years to make — versus 3–5 years later. (b) A direct test: does a film's own success predict how many similar stories follow it? |
| **3 · Network choices** | Do the plot-position results depend on choosing k = 20 neighbours and on the MiniLM embedding? | The network is rebuilt with k = 10, 20 and 30, and from a completely different text representation (TF-IDF word vectors), optionally also from a second sentence-BERT model. The Notebook 05 models and Moran's *I* are re-run on each network. |

| **4 · Plain units** | How large are the effects, in units a reader can picture? | Each effect restated as the difference between films at the 10th and 90th percentile of the measure, in percentile points, rating points or percent, next to cast star power. |
| **5 · Human check** | Do films with similar overviews really tell similar stories? | A blind rating sheet of linked, same-genre and random film pairs; once rated, the ratings are compared across the three kinds of pair. |
| **6 · Story vs stars** | Does a film's place in plot space matter more when the cast offers no star to sell it? | The Notebook 05 model (M4) plus each plot measure × has-a-star, fixed in advance: 4 measures × revenue percentile and rating. The 8 interaction tests are corrected as their own family (Benjamini–Hochberg). |
| **7 · Few clusters** | Standard errors are clustered by only 36 communities. Do the p-values of the results that survive the correction still hold? | Wild cluster bootstrap-t (restricted, Webb weights, 9,999 draws) for every result that survives the FDR correction in Parts 1 and 6, on the same models and films. |

Outputs go to `outputs/tables/robustness/` and `outputs/figures/robustness/`, with a notes file worded for the thesis. All results are associations, not causal effects.

**Run time:** about 15 minutes in Colab without a GPU (most of it is exact betweenness on the rebuilt networks). The optional second sentence-BERT model adds a few minutes on a GPU.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG
# ═══════════════════════════════════════════════════════════════════════════
CFG = {
    "project_dir": "/content/drive/MyDrive/thesis_movie_project",

    # ── Part 1: multiple testing ────────────────────────────────────────────
    "fdr_q": 0.05,                       # Benjamini–Hochberg false discovery rate

    # ── Part 2: foresight windows (years after release) ─────────────────────
    "foresight_windows": {"1 year after": (1, 1), "1–2 years after": (1, 2),
                          "3–5 years after": (3, 5), "1–5 years after (Notebook 06)": (1, 5)},
    "window_years": 5,                   # crowding window, as in Notebook 06
    "k_neighbours": 20,                  # as in Notebook 06
    "first_year": 1990, "last_year": 2024,
    "min_money_for_roi": 10_000,         # as in Notebook 06

    # ── Part 3: rebuilt networks ────────────────────────────────────────────
    "knn_values": [10, 20, 30],          # the main network uses k = 20
    "tfidf": {"min_df": 3, "max_df": 0.5, "svd_dims": 300},
    # A second sentence-BERT model. "auto" = run it only when a GPU is available (Colab: Runtime → Change
    # runtime type → GPU); True = always (slow on CPU, ~30 min); False = never.
    "second_sbert_model": "sentence-transformers/all-mpnet-base-v2",
    "run_second_sbert": "auto",
    "louvain_runs": 10,                  # communities of a rebuilt network: best of n Louvain runs
    "moran_permutations": 199,

    # ── Part 5: human similarity check ──────────────────────────────────────
    "validation_pairs_per_group": 30,    # linked / same genre / random → 90 pairs to rate (about an hour)
    "validation_min_votes": 50,          # films people may know, with proper overviews
    "validation_seed": 2026,
    "ai_raters": ["llm"],                # ratings files from an AI model, reported as such (never as human)
    "unblinded_pairs": {"llm": [14, 15]},  # pairs a rater saw in the key before rating — left out of that rater's analysis
    "random_state": 42,

    "figure_dpi": 200,
}
print("CONFIG loaded.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports, Drive mount, paths, style, helpers
# ═══════════════════════════════════════════════════════════════════════════
import json, os, random, shutil, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 100)
try:
    display
except NameError:
    display = print

try:
    import igraph as ig
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-igraph"], check=True)
    import igraph as ig

def mount_drive():
    """Mount Google Drive in Colab. Safe to re-run."""
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab — using local paths.")
        return
    if os.path.ismount("/content/drive"):
        print("Google Drive already mounted.")
        return
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        aside = f"/content/drive_local_leftover_{time.strftime('%Y%m%d_%H%M%S')}"
        shutil.move("/content/drive", aside)
        print(f"Moved a local leftover folder (not your Drive) to {aside}.")
    drive.mount("/content/drive")
mount_drive()

PROJECT_DIR = Path(os.environ.get("THESIS_PROJECT_DIR", CFG["project_dir"]))
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}. Check that Google Drive is mounted "
                            "(left sidebar → Files → drive/MyDrive) and that CFG['project_dir'] is right.")
NETWORK_DIR  = PROJECT_DIR / "processed" / "network"
FEATURES_DIR = PROJECT_DIR / "processed" / "features"
TABLES_DIR   = PROJECT_DIR / "outputs" / "tables"
R_TABLES     = TABLES_DIR / "robustness"
R_FIGS       = PROJECT_DIR / "outputs" / "figures" / "robustness"
for d in [R_TABLES, R_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

FILES = {
    "model_table":  FEATURES_DIR / "modeling_table_success_star_power.csv",
    "time_aware":   FEATURES_DIR / "time_aware_plot_features.csv",
    "masked_text":  FEATURES_DIR / "overview_person_names_masked.csv",
    "embeddings":   FEATURES_DIR / "sbert_minilm_embeddings_names_masked.npy",
    "emb_index":    FEATURES_DIR / "sbert_minilm_embedding_index_names_masked.csv",
    "nodes":        NETWORK_DIR / "semantic_network_nodes_lcc.csv",
    "regressions":  {y: TABLES_DIR / f for y, f in {
                        "log_revenue": "regression_results_log_revenue.csv",
                        "revenue_percentile_year": "regression_results_revenue_percentile.csv",
                        "vote_average": "regression_results_vote_average.csv",
                        "log_popularity": "regression_results_popularity.csv"}.items()},
    "model_fit":    TABLES_DIR / "success" / "model_fit_summary.csv",
    "timing":       TABLES_DIR / "narrative_dynamics" / "timing_regressions.csv",
    "communities":  TABLES_DIR / "success" / "fig_04_community_effects.csv",
}
flat = [p for v in FILES.values() for p in (v.values() if isinstance(v, dict) else [v])]
missing = [p.name for p in flat if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing}. Run Notebooks 03–06 v2 first.")

INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cd1"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 24,
    "axes.labelsize": 10, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "text.color": INK, "font.family": "DejaVu Sans", "lines.linewidth": 2, "legend.frameon": False,
})

def fig_header(fig, title, sub, band_in=0.85):
    """Lay out panels first, then put title and subtitle in a fixed-height band (inches) above them."""
    h = fig.get_figheight()
    fig.tight_layout()
    fig.subplots_adjust(top=1 - band_in / h)
    fig.text(0.01, 1 - 0.05 / h, title, fontsize=13, fontweight="bold", color=INK, va="top")
    fig.text(0.01, 1 - 0.37 / h, sub, fontsize=9.5, color=INK_2, va="top")

def save(fig, table, name):
    fig.savefig(R_FIGS / f"{name}.png", dpi=CFG["figure_dpi"], bbox_inches="tight")
    fig.savefig(R_FIGS / f"{name}.svg", bbox_inches="tight")
    table.to_csv(R_TABLES / f"{name}.csv", index=False)
    plt.show()
    print(f"  ✓ figures/robustness/{name}.png (+ .svg) + tables/robustness/{name}.csv")

def save_tab(df, name):
    df.to_csv(R_TABLES / name, index=False)
    print(f"  ✓ tables/robustness/{name}")

def fp(p):
    """p- or q-value for the notes (Notebooks 05–06 store p-values rounded to 5 decimals)."""
    return "< 0.0001" if p < 1e-4 else f"{p:.2g}"

def z(s):
    s = pd.to_numeric(s, errors="coerce")
    return (s - s.mean()) / s.std()

def fit(formula, data):
    """OLS with standard errors clustered by community — the same estimator as Notebooks 05–06."""
    return smf.ols(formula, data=data).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(data["community_group"])[0]})

OUTCOME_NAMES = {"log_revenue": "log revenue", "revenue_percentile_year": "revenue percentile",
                 "log_roi": "log ROI", "vote_average": "rating", "log_popularity": "log popularity"}
TERM_NAMES = {"z_novelty": "novelty", "z_novelty_sq": "novelty²", "z_log_betweenness": "betweenness",
              "z_participation": "participation", "z_clustering": "clustering",
              "crowding_z": "crowding", "I(crowding_z ** 2)": "crowding²", "foresight_z": "foresight",
              "competition_z": "same-year competition"}
NOTES = []
print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Load the modelling table (Notebook 05), time-aware features (Notebook 06) and embeddings (Notebook 03)
# ═══════════════════════════════════════════════════════════════════════════
df = pd.read_csv(FILES["model_table"], low_memory=False)
tf = pd.read_csv(FILES["time_aware"])
df = df.merge(tf[["tmdb_id", "crowding_z", "foresight_z", "competition_z"]], on="tmdb_id", how="left")
mm = CFG["min_money_for_roi"]
ok = (df["revenue_clean"] >= mm) & (df["budget_clean"] >= mm)
df["log_roi"] = np.where(ok, np.log(df["revenue_clean"].where(ok) / df["budget_clean"].where(ok)), np.nan)

E = np.load(FILES["embeddings"]).astype(np.float32)
E /= np.linalg.norm(E, axis=1, keepdims=True)
emb_index = pd.read_csv(FILES["emb_index"])
if len(emb_index) != len(E):
    raise ValueError("Embedding index does not match the embedding matrix — re-run Notebook 03 v2.")
YEARS = emb_index["release_year"].values
print(f"modelling table {df.shape} · embeddings {E.shape} · ROI for {int(ok.sum()):,} films")

# the Notebook 05 model, unchanged
CTRL = "C(release_year) + C(primary_genre_model) + z_runtime + is_english"
NET_TERMS = ["z_novelty", "z_log_betweenness", "z_participation", "z_clustering", "z_novelty_sq"]
STAR = "z_star_index + z_director_index + has_star"
COMM = "C(community_group, Treatment('minor'))"
NB05_OUTCOMES = ["log_revenue", "revenue_percentile_year", "vote_average", "log_popularity"]
NB05_VARS = ["release_year", "primary_genre_model", "z_runtime", "is_english", "z_star_index", "z_director_index",
             "has_star", "community_group"]

## Part 1 — Multiple testing

Notebook 05 tests 5 plot-position terms and Notebook 06 tests 4 timing terms, each on 4 outcomes: 36 tests. At p < 0.05, about 2 of 36 would come out "significant" even if nothing were going on. The **Benjamini–Hochberg** procedure controls the *false discovery rate*: among the results it keeps, at most 5% are expected to be false positives. Each test gets a *q*-value; a result survives if q < 0.05.

The 35 communities × 2 outcomes of Notebook 05 (70 tests) form a second family, corrected separately.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Benjamini–Hochberg over the plot/timing tests and over the community tests
# ═══════════════════════════════════════════════════════════════════════════
def bh(p):
    """Benjamini–Hochberg adjusted p-values (q-values)."""
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    q = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n)
    out[order] = np.minimum(q, 1)
    return out

reg = pd.concat([pd.read_csv(f) for f in FILES["regressions"].values()], ignore_index=True)
plot_tests = reg[(reg["model"] == "M4_add_community") & reg["term"].isin(NET_TERMS)].assign(source="Notebook 05 (M4)")
timing = pd.read_csv(FILES["timing"])
timing_tests = timing[timing["model"] == "T3_plus_competition"].assign(source="Notebook 06 (T3)")
tests = pd.concat([plot_tests, timing_tests], ignore_index=True)[["source", "outcome", "term", "coef", "ci_low", "ci_high", "p_value", "n_obs"]]
tests["q_value"] = bh(tests["p_value"])
tests["significant_raw"] = tests["p_value"] < 0.05
tests["survives_fdr"] = tests["q_value"] < CFG["fdr_q"]
tests["claim"] = tests["term"].map(TERM_NAMES) + " → " + tests["outcome"].map(OUTCOME_NAMES)
tests = tests.sort_values("p_value").reset_index(drop=True)
display(tests[tests["significant_raw"]][["claim", "coef", "p_value", "q_value", "survives_fdr"]].round(4))
save_tab(tests.round(6), "multiple_testing_plot_and_timing.csv")

ce = pd.read_csv(FILES["communities"])
comm = pd.concat([ce[["community_id", "label", f"{p}_coef", f"{p}_p_value"]]
                    .set_axis(["community_id", "label", "coef", "p_value"], axis=1).assign(outcome=o)
                  for p, o in [("revpct", "revenue_percentile_year"), ("rating", "vote_average")]], ignore_index=True)
comm["q_value"] = bh(comm["p_value"])
comm["significant_raw"] = comm["p_value"] < 0.05
comm["survives_fdr"] = comm["q_value"] < CFG["fdr_q"]
save_tab(comm.sort_values("p_value").round(6), "multiple_testing_communities.csv")

n_raw, n_fdr = int(tests["significant_raw"].sum()), int(tests["survives_fdr"].sum())
print(f"Plot position and timing: {n_raw} of {len(tests)} tests have p < 0.05; {n_fdr} survive the FDR correction "
      f"(about {0.05 * len(tests):.1f} false positives expected at p < 0.05 by chance alone).")
cs = comm.groupby("outcome")[["significant_raw", "survives_fdr"]].sum()
print("Communities differing from the average film (raw / after FDR):",
      {OUTCOME_NAMES[o]: f"{int(r.significant_raw)} / {int(r.survives_fdr)}" for o, r in cs.iterrows()})

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Figure: which plot and timing results survive the correction?
# ═══════════════════════════════════════════════════════════════════════════
show = tests[tests["significant_raw"] | tests["survives_fdr"]].sort_values("p_value", ascending=False)
fig, ax = plt.subplots(figsize=(9, 0.36 * len(show) + 1.9))
yy = np.arange(len(show))
P_FLOOR = 1e-5                                   # Notebooks 05–06 store p-values rounded to 5 decimals
logp = -np.log10(show["p_value"].clip(lower=P_FLOOR))
logq = -np.log10(show["q_value"].clip(lower=P_FLOOR))
ax.hlines(yy, logq, logp, color=AXIS, linewidth=2)
ax.plot(logp, yy, "o", color=MUTED, markersize=6, label="raw p")
ax.scatter(logq, yy, color=np.where(show["survives_fdr"], SERIES[0], SERIES[1]), s=45, zorder=3)
ax.axvline(-np.log10(0.05), color=INK_2, linewidth=1, linestyle="--")
ax.set_yticks(yy, [f"{c}  ({b:+.3f})" + ("  p < 0.00001" if p_ < P_FLOOR else "")
                   for c, b, p_ in zip(show["claim"], show["coef"], show["p_value"])])
ax.set_xlabel("−log₁₀ p   (right of the dashed line = below 0.05)")
ax.grid(axis="y", visible=False)
ax.legend(handles=[Line2D([], [], marker="o", color=MUTED, linestyle="", label="raw p-value"),
                   Line2D([], [], marker="o", color=SERIES[0], linestyle="", label="FDR q-value: survives"),
                   Line2D([], [], marker="o", color=SERIES[1], linestyle="", label="FDR q-value: does not survive")],
          loc="upper center", bbox_to_anchor=(0.5, -0.12 - 0.5 / fig.get_figheight()), ncol=3)
fig_header(fig, "Which plot and timing results survive a multiple-testing correction?",
           f"All results with p < 0.05 among {len(tests)} tests (Notebook 05 M4 plot terms, Notebook 06 T3 timing terms) · "
           "Benjamini–Hochberg, FDR 5% · coefficient in brackets")
save(fig, show[["claim", "coef", "p_value", "q_value", "survives_fdr"]], "fig_01_multiple_testing")

kept = tests[tests["survives_fdr"]]
lost = tests[tests["significant_raw"] & ~tests["survives_fdr"]]
NOTES.append(f"Multiple testing: {n_raw} of {len(tests)} plot-position and timing tests have p < 0.05 "
             f"(≈{0.05 * len(tests):.0f} expected by chance); {n_fdr} survive a Benjamini–Hochberg correction (FDR 5%).")
if len(kept):
    NOTES.append("  Survive: " + "; ".join(f"{r.claim} ({r.coef:+.3f}, q {fp(r.q_value)})" for r in kept.itertuples()) + ".")
if len(lost):
    NOTES.append("  Do not survive (report as tentative): " + "; ".join(f"{r.claim} (p {fp(r.p_value)}, q {fp(r.q_value)})" for r in lost.itertuples()) + ".")
NOTES.append("Communities differing from the average film after controls, raw → after FDR: "
             + "; ".join(f"{OUTCOME_NAMES[o]} {int(r.significant_raw)} → {int(r.survives_fdr)} of 35" for o, r in cs.iterrows()) + ".")

## Part 2 — Foresight or imitation?

Notebook 06 measured **foresight** as how much *later* films (1–5 years after release) resemble a film, minus how much *earlier* films (1–5 years before) did. Films with high foresight did better on all four outcomes.

The obvious objection: **successful films get copied.** A hit inspires similar films, so success could cause foresight rather than the other way round. Two tests:

**(a) Windows.** A film released one or two years after another was already in production when the first came out, so it cannot be a copy of its success. If foresight still predicts success when it is measured only against films released 1–2 years later, imitation cannot explain it. If the association only appears 3–5 years later, imitation remains possible.

**(b) Does success attract similar stories?** If hits get copied, a film's own success should predict how many similar stories appear 3–5 years later (when copies can arrive) more than 1–2 years later (when they cannot). The film's crowding, year, genre, community and star power are held constant.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — Echo and foresight for several windows (same definition as Notebook 06)
# ═══════════════════════════════════════════════════════════════════════════
K, W = CFG["k_neighbours"], CFG["window_years"]
WINDOWS = CFG["foresight_windows"]
max_ahead = max(b for _, b in WINDOWS.values())

def topk_mean(q, pool):
    S = E[q] @ E[pool].T
    return np.partition(S, -K, axis=1)[:, -K:].mean(axis=1)

crowd = np.full(len(E), np.nan)
echo = {w: np.full(len(E), np.nan) for w in WINDOWS}
for y in np.unique(YEARS):
    q = np.where(YEARS == y)[0]
    if y - W >= CFG["first_year"]:
        crowd[q] = topk_mean(q, np.where((YEARS >= y - W) & (YEARS <= y - 1))[0])
    if y + max_ahead <= CFG["last_year"]:            # the same films for every window
        for w, (a, b) in WINDOWS.items():
            echo[w][q] = topk_mean(q, np.where((YEARS >= y + a) & (YEARS <= y + b))[0])

tw = emb_index[["tmdb_id", "release_year"]].copy()
cols = {"crowding_check": crowd}
for i, w in enumerate(WINDOWS):
    cols[f"echo_{i}"], cols[f"foresight_{i}"] = echo[w], echo[w] - crowd
for c, v in cols.items():
    tw[c] = v
    g = tw.groupby("release_year")[c]
    tw[c] = (tw[c] - g.transform("mean")) / g.transform("std")       # within-year z, as in Notebook 06
WKEY = {w: i for i, w in enumerate(WINDOWS)}
d2 = df.merge(tw.drop(columns="release_year"), on="tmdb_id", how="left")
full = WKEY["1–5 years after (Notebook 06)"]
both_ = d2[["foresight_z", f"foresight_{full}"]].dropna()
r_check = both_.corr().iloc[0, 1]
print(f"Check: 1–5-year foresight here vs Notebook 06: r = {r_check:.4f} on {len(both_):,} films (expected 1.000)")
print("Correlation between the foresight windows:")
display(d2[[f"foresight_{i}" for i in WKEY.values()]].set_axis(list(WINDOWS), axis=1).corr().round(2))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — (a) Foresight → success, by window  ·  (b) success → later similar stories
# ═══════════════════════════════════════════════════════════════════════════
BASE = ("C(release_year) + C(primary_genre_model) + z_runtime + is_english + z_star_index + z_director_index + has_star "
        "+ C(community_group)")                      # Notebook 06 controls
TIMING_OUT = ["revenue_percentile_year", "log_roi", "vote_average", "log_popularity"]
need = ["release_year", "primary_genre_model", "z_runtime", "is_english", "z_star_index", "z_director_index",
        "has_star", "community_group", "crowding_z", "competition_z"]
win_rows, imit_rows = [], []
for y in TIMING_OUT:
    d = d2.dropna(subset=[y] + need + [f"foresight_{i}" for i in WKEY.values()])     # one sample for all windows
    for w, i in WKEY.items():
        r = fit(f"{y} ~ {BASE} + crowding_z + I(crowding_z**2) + competition_z + foresight_{i}", d)
        t = f"foresight_{i}"
        win_rows.append({"outcome": y, "window": w, "coef": r.params[t], "ci_low": r.conf_int().loc[t, 0],
                         "ci_high": r.conf_int().loc[t, 1], "p_value": r.pvalues[t], "n_obs": int(r.nobs)})
    if y == "log_roi":
        continue
    d = d.assign(success_z=z(d[y]))
    for w in ["1 year after", "1–2 years after", "3–5 years after"]:
        r = fit(f"echo_{WKEY[w]} ~ {BASE} + crowding_z + success_z", d)
        imit_rows.append({"outcome": y, "window": w, "coef": r.params["success_z"],
                          "ci_low": r.conf_int().loc["success_z", 0], "ci_high": r.conf_int().loc["success_z", 1],
                          "p_value": r.pvalues["success_z"], "n_obs": int(r.nobs)})
win = pd.DataFrame(win_rows)
imit = pd.DataFrame(imit_rows)
display(win.round(4)); display(imit.round(4))
save_tab(win.round(6), "foresight_by_window.csv")
save_tab(imit.round(6), "success_predicts_later_similar_stories.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Figure: foresight by window and the imitation test
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, len(TIMING_OUT) + 1, figsize=(17, 3.9),
                         gridspec_kw={"width_ratios": [1] * len(TIMING_OUT) + [1.25]})
wins = list(WINDOWS)
wcol = [SERIES[0], SERIES[0], SERIES[1], INK_2]
for ax, y in zip(axes, TIMING_OUT):
    t = win[win["outcome"] == y].set_index("window").reindex(wins)
    yy = np.arange(len(wins))[::-1]
    sig = (t["p_value"] < 0.05).values
    for j in range(len(wins)):
        ax.hlines(yy[j], t["ci_low"].iloc[j], t["ci_high"].iloc[j], color=wcol[j] if sig[j] else AXIS, linewidth=3, alpha=0.8)
        ax.plot(t["coef"].iloc[j], yy[j], "o", color=wcol[j], markersize=5)
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_title(OUTCOME_NAMES[y], fontsize=11, pad=8)
    ax.set_yticks(yy, wins if ax is axes[0] else [""] * len(wins))
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("effect of 1 SD foresight")
ax = axes[-1]
ow = ["1 year after", "1–2 years after", "3–5 years after"]
outs = [o for o in TIMING_OUT if o != "log_roi"]
for k_, y in enumerate(outs):
    t = imit[imit["outcome"] == y].set_index("window").reindex(ow)
    xx = np.arange(len(ow)) + (k_ - 1) * 0.22
    ax.errorbar(xx, t["coef"], yerr=[t["coef"] - t["ci_low"], t["ci_high"] - t["coef"]], fmt="o", color=[SERIES[2], SERIES[3], SERIES[4]][k_],
                markersize=5, capsize=0, elinewidth=2, label=OUTCOME_NAMES[y])
ax.axhline(0, color=INK_2, linewidth=1)
ax.set_xticks(range(len(ow)), ["1 yr", "1–2 yrs", "3–5 yrs"])
ax.set_xlabel("similar stories released … after the film")
ax.set_ylabel("change in later similarity per 1 SD success")
ax.set_title("Does success attract similar stories?", fontsize=11, pad=8)
ax.legend(loc="upper left", fontsize=8.5)
fig_header(fig, "Ahead of its time, or copied because it succeeded?",
           "Left: effect of foresight measured against films released 1, 1–2, 3–5 and 1–5 years later (same films, Notebook 06 controls) · "
           "right: own success → similarity of later films · 95% CI clustered by community")
save(fig, pd.concat([win.assign(panel="foresight by window"), imit.assign(panel="success → later similar stories")]),
     "fig_02_foresight_or_imitation")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Notes for Part 2 (written from the numbers)
# ═══════════════════════════════════════════════════════════════════════════
def sig_list(rows):
    return ", ".join(OUTCOME_NAMES[r.outcome] for r in rows.itertuples()) or "none"

early = win[win["window"] == "1–2 years after"]
late = win[win["window"] == "3–5 years after"]
fullw = win[win["window"] == "1–5 years after (Notebook 06)"]
NOTES.append(f"Foresight windows (same {early['n_obs'].min():,}–{early['n_obs'].max():,} films per outcome, Notebook 06 controls): "
             f"significant with the 1–5-year window for {sig_list(fullw[fullw.p_value < 0.05])}; "
             f"with films released 1–2 years later (too soon to be copies) for {sig_list(early[early.p_value < 0.05])}; "
             f"3–5 years later for {sig_list(late[late.p_value < 0.05])}.")
NOTES.append("  1–2-year window: " + "; ".join(f"{OUTCOME_NAMES[r.outcome]} {r.coef:+.4f} (p={r.p_value:.2g})" for r in early.itertuples())
             + ". 3–5-year window: " + "; ".join(f"{OUTCOME_NAMES[r.outcome]} {r.coef:+.4f} (p={r.p_value:.2g})" for r in late.itertuples()) + ".")
for y in outs:
    t = imit[imit["outcome"] == y].set_index("window")
    e_, l_ = t.loc["1–2 years after"], t.loc["3–5 years after"]
    verdict = ("imitation signature: success predicts similar stories 3–5 years later"
               if l_["p_value"] < 0.05 and l_["coef"] > 0 and l_["coef"] > e_["coef"] else
               "no imitation signature: success does not predict more similar stories 3–5 years later")
    NOTES.append(f"Imitation test, {OUTCOME_NAMES[y]}: 1 SD more success → later similarity {e_['coef']:+.4f} SD after 1–2 years "
                 f"(p={e_['p_value']:.2g}), {l_['coef']:+.4f} SD after 3–5 years (p={l_['p_value']:.2g}) — {verdict}.")
NOTES.append(f"Check: the 1–5-year foresight measure reproduces Notebook 06 (r = {r_check:.4f}).")
print("\n".join(NOTES[-(len(outs) + 3):]))

## Part 3 — Do the plot-position results depend on how the network was built?

The main network links each film to its **k = 20** most similar plots (mutual neighbours only), using **MiniLM** sentence embeddings. Here the network is rebuilt from scratch with other choices, and the Notebook 05 models (same controls, same films where possible) and Moran's *I* are re-run on each:

| Network | Text representation | k |
|---|---|---|
| MiniLM k = 10 / 20 / 30 | the main embeddings (Notebook 03) | 10, 20, 30 |
| TF-IDF k = 20 | word counts weighted by rarity, reduced to 300 dimensions — no neural model at all | 20 |
| mpnet k = 20 (optional) | a second, larger sentence-BERT model | 20 |

For each network: novelty (mean distance to the k most similar plots), exact betweenness, participation (on that network's own Louvain communities) and weighted clustering are recomputed. The **community control** in the models stays the 35 labelled communities of the main network, so only the plot-position measures change. *MiniLM k = 20* rebuilds the main network; it differs only in using best-of-10 Louvain instead of consensus communities for participation, and shows that the rebuild reproduces Notebook 05.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Text representations: MiniLM (main), TF-IDF, optional second sentence-BERT model
# ═══════════════════════════════════════════════════════════════════════════
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

texts = (emb_index[["tmdb_id"]].merge(pd.read_csv(FILES["masked_text"], usecols=["tmdb_id", "overview_masked"])
                                      .drop_duplicates("tmdb_id"), on="tmdb_id", how="left")["overview_masked"].fillna(""))
print(f"Masked overviews matched: {(texts != '').sum():,} of {len(texts):,}")

REPS = {"MiniLM": E}
t0 = time.time()
tcfg = CFG["tfidf"]
X = TfidfVectorizer(stop_words="english", min_df=tcfg["min_df"], max_df=tcfg["max_df"], sublinear_tf=True).fit_transform(texts)
T = TruncatedSVD(tcfg["svd_dims"], random_state=CFG["random_state"]).fit_transform(X).astype(np.float32)
T /= np.maximum(np.linalg.norm(T, axis=1, keepdims=True), 1e-9)
REPS["TF-IDF"] = T
print(f"TF-IDF: {X.shape[1]:,} words → {T.shape[1]} dimensions in {time.time() - t0:.0f}s")

def want_second_model():
    flag = CFG["run_second_sbert"]
    if flag is False or not CFG["second_sbert_model"]:
        return False
    try:
        import torch
        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False
    return flag is True or gpu

if want_second_model():
    try:
        from sentence_transformers import SentenceTransformer
        t0 = time.time()
        name = CFG["second_sbert_model"]
        M = SentenceTransformer(name).encode(texts.tolist(), batch_size=64, show_progress_bar=True,
                                             convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
        REPS[name.split("/")[-1]] = M
        print(f"{name}: {M.shape} in {(time.time() - t0) / 60:.1f} min")
    except Exception as ex:
        print(f"Second sentence-BERT model skipped ({type(ex).__name__}: {ex}).")
else:
    print("Second sentence-BERT model not run (no GPU; set CFG['run_second_sbert'] = True to run it on CPU).")

VARIANTS = [("MiniLM", k) for k in CFG["knn_values"]] + [(r, 20) for r in REPS if r != "MiniLM"]
print("Networks to build:", [f"{r} k={k}" for r, k in VARIANTS])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Build each network and recompute the plot-position measures
# ═══════════════════════════════════════════════════════════════════════════
def knn(M, k, block=2000):
    """Indices and cosine similarities of the k most similar films (self excluded)."""
    n = len(M)
    ind, sim = np.empty((n, k), np.int64), np.empty((n, k), np.float32)
    for s in range(0, n, block):
        S = M[s:s + block] @ M.T
        S[np.arange(S.shape[0]), np.arange(s, s + S.shape[0])] = -np.inf
        top = np.argpartition(S, -k, axis=1)[:, -k:]
        ind[s:s + block], sim[s:s + block] = top, np.take_along_axis(S, top, axis=1)
    return ind, sim

def build_network(M, k):
    n = len(M)
    ind, sim = knn(M, k)
    d = pd.DataFrame({"a": np.repeat(np.arange(n), k), "b": ind.ravel(), "w": sim.ravel()})
    e = d[d["a"] < d["b"]].merge(d.rename(columns={"a": "b", "b": "a", "w": "w2"}), on=["a", "b"])   # mutual only
    e["w"] = (e["w"] + e["w2"]) / 2
    g = ig.Graph(n=n, edges=e[["a", "b"]].values.tolist())
    g.es["weight"] = e["w"].values
    memb = np.array(g.connected_components().membership)
    keep = np.where(memb == np.bincount(memb).argmax())[0]           # largest connected component
    h = g.induced_subgraph(keep)
    N = h.vcount()

    best = None                                                       # communities: best of n Louvain runs
    for s in range(CFG["louvain_runs"]):
        random.seed(CFG["random_state"] + s)
        ig.set_random_number_generator(random)
        c = h.community_multilevel(weights="weight")
        if best is None or c.modularity > best.modularity:
            best = c
    comm = np.array(best.membership)

    betw = np.array(h.betweenness(directed=False)) * 2 / ((N - 1) * (N - 2))      # networkx 'normalized' scale
    el = np.array(h.get_edgelist())
    w = np.array(h.es["weight"])
    G = nx.Graph()
    G.add_nodes_from(range(N))
    G.add_weighted_edges_from(zip(el[:, 0], el[:, 1], w))
    clust = nx.clustering(G, weight="weight")

    # participation coefficient (Guimerà & Amaral), weighted, as in Notebook 04
    both = pd.DataFrame({"node": np.r_[el[:, 0], el[:, 1]], "c": np.r_[comm[el[:, 1]], comm[el[:, 0]]], "w": np.r_[w, w]})
    k_is = both.groupby(["node", "c"])["w"].sum()
    k_i = k_is.groupby(level=0).sum()
    part = 1 - ((k_is / k_i.reindex(k_is.index.get_level_values(0)).values) ** 2).groupby(level=0).sum()

    novelty = 1 - sim.mean(axis=1)                                    # mean cosine distance to the k most similar plots
    feats = pd.DataFrame({"tmdb_id": emb_index["tmdb_id"].values[keep],
                          "novelty": novelty[keep], "log_betweenness": np.log1p(betw * N),
                          "participation": part.reindex(range(N)).fillna(0).values,
                          "clustering": pd.Series(clust).reindex(range(N)).values})
    edges_t = pd.DataFrame({"a": emb_index["tmdb_id"].values[keep][el[:, 0]], "b": emb_index["tmdb_id"].values[keep][el[:, 1]]})
    info = {"films_in_lcc": N, "edges": h.ecount(), "mean_degree": 2 * h.ecount() / N,
            "communities": len(best), "modularity": best.modularity}
    return feats, edges_t, info

NETS = {}
for rep, k in VARIANTS:
    t0 = time.time()
    feats, edges_t, info = build_network(REPS[rep], k)
    NETS[(rep, k)] = (feats, edges_t, info)
    print(f"{rep:7s} k={k:2d}: {info['films_in_lcc']:,} films, {info['edges']:,} edges, mean degree {info['mean_degree']:.1f}, "
          f"{info['communities']} communities (modularity {info['modularity']:.3f}) · {time.time() - t0:.0f}s")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Re-run the Notebook 05 models and Moran's I on each network
# ═══════════════════════════════════════════════════════════════════════════
fitsum = pd.read_csv(FILES["model_fit"])
base_cols = [c for c in df.columns if c not in ("novelty", "log_betweenness", "participation", "clustering")
                           and not c.startswith(("z_novelty", "z_log_betweenness", "z_participation", "z_clustering"))]
rng = np.random.default_rng(CFG["random_state"])

def morans_i(d, y, edges_t, n_perm):
    """Moran's I of the outcome adjusted for year and genre (as in Notebook 06), with a permutation p-value."""
    d = d.dropna(subset=[y]).reset_index(drop=True)
    v = smf.ols(f"{y} ~ C(release_year) + C(primary_genre_model)", d).fit().resid.values
    v = (v - v.mean()) / v.std()
    pos = pd.Series(np.arange(len(d)), index=d["tmdb_id"].values)
    a, b = edges_t["a"].map(pos), edges_t["b"].map(pos)
    m = a.notna() & b.notna()
    a, b = a[m].astype(int).values, b[m].astype(int).values
    stat = lambda x: (len(x) / len(a)) * np.sum(x[a] * x[b]) / np.sum(x ** 2)
    obs = stat(v)
    null = np.array([stat(rng.permutation(v)) for _ in range(n_perm)])
    return obs, (np.sum(null >= obs) + 1) / (n_perm + 1), np.quantile(null, 0.95)

rob_rows, fit_rows = [], []
for (rep, k), (feats, edges_t, info) in NETS.items():
    v = df[base_cols].merge(feats, on="tmdb_id", how="inner")
    for c in ["novelty", "log_betweenness", "participation", "clustering"]:
        v[f"z_{c}"] = z(v[c])
    v["z_novelty_sq"] = v["z_novelty"] ** 2
    net = " + ".join(NET_TERMS)
    for y in NB05_OUTCOMES:
        d = v.dropna(subset=[y] + NB05_VARS + NET_TERMS)
        m1 = fit(f"{y} ~ {CTRL}", d)
        m2 = fit(f"{y} ~ {CTRL} + {net}", d)
        m4 = fit(f"{y} ~ {CTRL} + {net} + {STAR} + {COMM}", d)
        wald = m4.wald_test(" , ".join(f"{t} = 0" for t in NET_TERMS), scalar=True)
        fit_rows.append({"network": f"{rep} k={k}", "representation": rep, "k": k, "outcome": y, "n_obs": int(m4.nobs),
                         "r2_controls": m1.rsquared, "r2_gain_plot_network": m2.rsquared - m1.rsquared,
                         "plot_block_p_M4": float(wald.pvalue), **info})
        ci = m4.conf_int()
        for t in NET_TERMS:
            rob_rows.append({"network": f"{rep} k={k}", "representation": rep, "k": k, "outcome": y, "term": t,
                             "coef": m4.params[t], "ci_low": ci.loc[t, 0], "ci_high": ci.loc[t, 1],
                             "p_value": m4.pvalues[t], "n_obs": int(m4.nobs)})
    for y in ["revenue_percentile_year", "vote_average"]:
        I_, p_, p95 = morans_i(v, y, edges_t, CFG["moran_permutations"])
        fit_rows.append({"network": f"{rep} k={k}", "representation": rep, "k": k, "outcome": y,
                         "morans_I": I_, "morans_p": p_, "morans_null_p95": p95, **info})
    print(f"{rep} k={k} done")

rob = pd.DataFrame(rob_rows)
nf = pd.DataFrame(fit_rows)
fits_ = nf[nf["r2_controls"].notna()].drop(columns=["morans_I", "morans_p", "morans_null_p95"])
mor = nf[nf["morans_I"].notna()][["network", "representation", "k", "outcome", "morans_I", "morans_p", "morans_null_p95"]]
networks = fits_.drop_duplicates("network")[["network", "representation", "k", "films_in_lcc", "edges", "mean_degree", "communities", "modularity"]]
display(networks.round(3))
display(fits_.pivot(index="network", columns="outcome", values="r2_gain_plot_network").round(4))
display(mor.pivot(index="network", columns="outcome", values="morans_I").round(3))
save_tab(networks.round(5), "rebuilt_networks.csv")
save_tab(fits_.round(6), "network_robustness_model_fit.csv")
save_tab(rob.round(6), "network_robustness_coefficients.csv")
save_tab(mor.round(6), "network_robustness_morans_i.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Figure: plot-position coefficients across networks
# ═══════════════════════════════════════════════════════════════════════════
nets = list(dict.fromkeys(rob["network"]))
main_net = "MiniLM k=20"
ncol = {n: c for n, c in zip(nets, [SERIES[0], INK, SERIES[2], SERIES[1], SERIES[4]])}
terms = NET_TERMS
fig, axes = plt.subplots(1, len(NB05_OUTCOMES), figsize=(17, 0.75 * len(terms) + 2.2), sharey=True)
off = np.linspace(-0.3, 0.3, len(nets))
for ax, y in zip(axes, NB05_OUTCOMES):
    for j, n in enumerate(nets):
        t = rob[(rob["outcome"] == y) & (rob["network"] == n)].set_index("term").reindex(terms)
        yy = np.arange(len(terms))[::-1] + off[j]
        sig = (t["p_value"] < 0.05).values
        ax.hlines(yy, t["ci_low"], t["ci_high"], color=[ncol[n] if s else AXIS for s in sig], linewidth=2, alpha=0.8)
        ax.scatter(t["coef"], yy, color=ncol[n], s=22 if n != main_net else 34, zorder=3,
                   marker="D" if n == main_net else "o", label=n if y == NB05_OUTCOMES[0] else None)
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_title(OUTCOME_NAMES[y], fontsize=11, pad=8)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(np.arange(len(terms))[::-1], [TERM_NAMES[t] for t in terms])
fig.legend(loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=len(nets))
fig_header(fig, "Do the plot-position results depend on how the network was built?",
           "Notebook 05 M4 coefficients (per 1 SD) recomputed on networks rebuilt with other k and another text representation · "
           "95% CI clustered by community · grey = p ≥ 0.05 · ◆ = rebuild of the main network")
fig.subplots_adjust(bottom=0.2)
save(fig, rob, "fig_03_network_robustness")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — Notes for Part 3 (written from the numbers)
# ═══════════════════════════════════════════════════════════════════════════
main_gain = fitsum[fitsum["model"].isin(["M1_controls", "M2_add_plot_network"])].pivot(
    index="outcome", columns="model", values="r_squared")
main_gain = (main_gain["M2_add_plot_network"] - main_gain["M1_controls"]).to_dict()
g = fits_.pivot(index="network", columns="outcome", values="r2_gain_plot_network")
NOTES.append("Rebuilt networks: " + "; ".join(f"{r.network} {r.films_in_lcc:,} films, mean degree {r.mean_degree:.1f}"
                                              for r in networks.itertuples()) + ".")
NOTES.append(f"R² added by plot position over the controls ranges {g.values.min():.4f}–{g.values.max():.4f} across "
             f"{len(g)} networks and 4 outcomes (main network: {min(main_gain.values()):.4f}–{max(main_gain.values()):.4f}).")
pb = fits_.pivot(index="network", columns="outcome", values="plot_block_p_M4")
NOTES.append("Plot-position block jointly significant in M4 (p < 0.05): " + "; ".join(
    f"{OUTCOME_NAMES[o]} in {int((pb[o] < 0.05).sum())} of {len(pb)} networks" for o in NB05_OUTCOMES) + ".")

# sign stability for the results Notebook 05 reports as significant
main_sig = plot_tests[plot_tests["p_value"] < 0.05]
for r in main_sig.itertuples():
    t = rob[(rob["outcome"] == r.outcome) & (rob["term"] == r.term)]
    same = t[(np.sign(t["coef"]) == np.sign(r.coef)) & (t["p_value"] < 0.05)]["network"].tolist()
    weak = t[(np.sign(t["coef"]) == np.sign(r.coef)) & (t["p_value"] >= 0.05)]["network"].tolist()
    flip = t[np.sign(t["coef"]) != np.sign(r.coef)]["network"].tolist()
    NOTES.append(f"  {TERM_NAMES[r.term]} → {OUTCOME_NAMES[r.outcome]} (main {r.coef:+.3f}): same sign and p < 0.05 in "
                 f"{', '.join(same) or 'none'}" + (f"; same sign, not significant in {', '.join(weak)}" if weak else "")
                 + (f"; opposite sign in {', '.join(flip)}" if flip else "") + ".")
mr = mor.pivot(index="network", columns="outcome", values="morans_I")
mp = mor.pivot(index="network", columns="outcome", values="morans_p")
NOTES.append("Moran's I (similar plots, similar fates) across networks: " + "; ".join(
    f"{OUTCOME_NAMES[o]} {mr[o].min():.3f}–{mr[o].max():.3f}, p < 0.05 in {int((mp[o] < 0.05).sum())} of {len(mp)}"
    for o in mr.columns) + ".")
print("\n".join(NOTES[-(len(main_sig) + 4):]))

## Part 4 — What the effects mean in plain units

Notebooks 05–06 report coefficients per standard deviation of a predictor, in the units of each outcome (a revenue *share*, log popularity, …). Here every plot and timing result is restated as the difference between a film at the **10th and the 90th percentile** of that predictor, all else equal, in units a reader can picture:

| Outcome | Plain unit |
|---|---|
| revenue percentile | percentile points of the year's market, and roughly what that means in revenue for a film in the middle of the market |
| rating | TMDB rating points (scale 0–10) |
| popularity, revenue, ROI | percent difference |

As a yardstick, the same is shown for **cast star power** (Notebook 05).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 16 — Effects from the 10th to the 90th percentile of each predictor, in plain units
# ═══════════════════════════════════════════════════════════════════════════
PRED_COL = {"z_log_betweenness": "z_log_betweenness", "z_participation": "z_participation", "z_clustering": "z_clustering",
            "z_novelty": "z_novelty", "crowding_z": "crowding_z", "foresight_z": "foresight_z",
            "competition_z": "competition_z", "z_star_index": "z_star_index"}
spread = {t: df[c].quantile(0.9) - df[c].quantile(0.1) for t, c in PRED_COL.items()}      # P10 → P90, in SD units

# market revenue per year (all thesis films with revenue), to turn percentile points into a revenue difference
thesis_rev = pd.read_csv(PROJECT_DIR / "processed" / "core" / "df_thesis_films.csv", usecols=["release_year", "revenue_clean"])
thesis_rev = thesis_rev[pd.to_numeric(thesis_rev["revenue_clean"], errors="coerce") > 0]

def pct_points_to_revenue(dp):
    """Revenue ratio between percentile 0.5 + dp/2 and 0.5 − dp/2 of a year's market (median over 1995–2019)."""
    r = []
    for y, g in thesis_rev[thesis_rev["release_year"].between(1995, 2019)].groupby("release_year"):
        lo, hi = g["revenue_clean"].quantile([0.5 - dp / 2, 0.5 + dp / 2])
        r.append(hi / lo - 1)
    return float(np.median(r))

def plain(outcome, x):
    """Convert an effect x (outcome units) into the plain unit."""
    if outcome == "revenue_percentile_year":
        return x * 100
    if outcome == "vote_average":
        return x
    return (np.exp(x) - 1) * 100            # log outcomes → percent

UNIT = {"revenue_percentile_year": "percentile points", "vote_average": "rating points (0–10)",
        "log_popularity": "% popularity", "log_revenue": "% revenue", "log_roi": "% return on investment"}

star = reg[(reg["model"] == "M4_add_community") & (reg["term"] == "z_star_index")].assign(source="Notebook 05 (M4)")
eff = pd.concat([tests[tests["term"].isin(PRED_COL)], star], ignore_index=True)
eff = eff[eff["outcome"].isin(UNIT)].copy()
eff["p10_p90_sd"] = eff["term"].map(spread)
for c in ["coef", "ci_low", "ci_high"]:
    eff[f"plain_{c}"] = [plain(o, v * s) for o, v, s in zip(eff["outcome"], eff[c], eff["p10_p90_sd"])]
eff["unit"] = eff["outcome"].map(UNIT)
eff["claim"] = eff["term"].map({**TERM_NAMES, "z_star_index": "cast star power"}) + " → " + eff["outcome"].map(OUTCOME_NAMES)
eff["status"] = np.where(eff["term"] == "z_star_index", "benchmark",
                np.where(eff["survives_fdr"].fillna(False).astype(bool), "survives FDR",
                np.where(eff["p_value"] < 0.05, "tentative", "not significant")))
rp = eff["outcome"] == "revenue_percentile_year"
def revenue_equivalent(v):
    """Percentile-point effect v → % revenue difference for a film in the middle of its year's market."""
    r = pct_points_to_revenue(abs(v) / 100)            # higher film earns (1 + r) × the lower one
    return 100 * (r if v >= 0 else 1 / (1 + r) - 1)
eff.loc[rp, "revenue_equivalent_pct"] = [revenue_equivalent(v) for v in eff.loc[rp, "plain_coef"]]
show_cols = ["claim", "status", "plain_coef", "plain_ci_low", "plain_ci_high", "unit", "revenue_equivalent_pct", "p_value", "q_value"]
eff = eff.sort_values(["outcome", "status", "p_value"]).reset_index(drop=True)
display(eff[eff["status"] != "not significant"][show_cols].round(3))
save_tab(eff[["source", "outcome", "term", "claim", "status", "coef", "p10_p90_sd"] + show_cols[2:]].round(5), "effects_in_plain_units.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 17 — Figure: effects in plain units, next to star power
# ═══════════════════════════════════════════════════════════════════════════
fig_out = [o for o in ["revenue_percentile_year", "vote_average", "log_popularity", "log_revenue", "log_roi"]
           if ((eff["outcome"] == o) & eff["status"].isin(["survives FDR", "tentative"])).any()]
STATUS_COL = {"survives FDR": SERIES[0], "tentative": SERIES[1], "benchmark": INK_2}
STATUS_LABEL = {"survives FDR": "survives FDR correction", "tentative": "tentative (p < 0.05, not after correction)",
                "benchmark": "cast star power (yardstick)"}
rows_per = [((eff["outcome"] == o) & (eff["status"] != "not significant")).sum() for o in fig_out]
fig, axes = plt.subplots(len(fig_out), 1, figsize=(10, 0.42 * sum(rows_per) + 1.1 * len(fig_out) + 1.2),
                         gridspec_kw={"height_ratios": rows_per})
fig_tab = []
for ax, o in zip(np.atleast_1d(axes), fig_out):
    t = eff[(eff["outcome"] == o) & (eff["status"] != "not significant")].sort_values("plain_coef")
    yy = np.arange(len(t))
    cols_ = [STATUS_COL[s] for s in t["status"]]
    ax.hlines(yy, t["plain_ci_low"], t["plain_ci_high"], color=cols_, linewidth=3, alpha=0.45)
    ax.scatter(t["plain_coef"], yy, color=cols_, s=40, zorder=3)
    for y_, v, s in zip(yy, t["plain_coef"], t["status"]):
        ax.annotate(f"{v:+.2f}", (v, y_), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8, color=INK_2)
    ax.set_yticks(yy, [c.split(" → ")[0] for c in t["claim"]])
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_title(f"{OUTCOME_NAMES[o]} · {UNIT[o]}", fontsize=10.5, pad=6, loc="left")
    ax.grid(axis="y", visible=False)
    ax.set_ylim(-0.7, len(t) - 0.3)
    fig_tab.append(t)
fig.legend(handles=[Line2D([], [], marker="o", color=c, linestyle="", label=STATUS_LABEL[l]) for l, c in STATUS_COL.items()],
           loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01))
fig_header(fig, "How large are the effects? From the 10th to the 90th percentile of each measure",
           "Difference between films at the 10th and 90th percentile of the measure, all else equal (Notebook 05 M4 / Notebook 06 T3) · "
           "95% CI · star power as yardstick", band_in=0.9)
fig.subplots_adjust(bottom=0.9 / fig.get_figheight(), hspace=0.9)
save(fig, pd.concat(fig_tab)[show_cols], "fig_04_effects_in_plain_units")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 18 — Notes for Part 4
# ═══════════════════════════════════════════════════════════════════════════
def plain_sentence(r):
    s = f"{r.claim}: {r.plain_coef:+.2f} {r.unit} [{r.plain_ci_low:+.2f}, {r.plain_ci_high:+.2f}]"
    if pd.notna(r.revenue_equivalent_pct):
        s += f" (≈ {r.revenue_equivalent_pct:+.0f}% revenue for a film in the middle of its year's market)"
    return s

NOTES.append("Effects in plain units (film at the 90th vs the 10th percentile of the measure, all else equal):")
for status in ["survives FDR", "tentative"]:
    t = eff[eff["status"] == status]
    if len(t):
        NOTES.append(f"  {status}: " + "; ".join(plain_sentence(r) for r in t.itertuples()) + ".")
b = eff[eff["status"] == "benchmark"]
NOTES.append("  Yardstick, cast star power: " + "; ".join(plain_sentence(r) for r in b.itertuples())
             + ". This is the cast star index alone; director reach and the has-a-star indicator carry part of star power "
               "too, so the yardstick understates star power as a whole.")
print("\n".join(NOTES[-4:]))

## Part 5 — Do films with similar overviews really tell similar stories? (blind rating check)

The whole thesis rests on one assumption: when two overviews have similar embeddings, the films tell similar stories. This part checks it with human judgement.

**Step 1 (automatic).** The notebook draws film pairs of three kinds and writes them, shuffled and **without titles or pair type**, to a rating sheet:
* **linked** — pairs joined by an edge in the plot network;
* **same genre** — random pairs from the same primary genre that are *not* linked (the harder comparison: does the network capture more than genre?);
* **random** — random unlinked pairs.

**Step 2 (you).** Open `outputs/tables/robustness/similarity_rating_sheet.csv` (Google Sheets or Excel), read the two overviews in each row and fill the column `rating` from **1** (completely different stories) to **5** (essentially the same kind of story). Do not open `similarity_rating_key.csv`. Save it as `similarity_ratings_rater1.csv` in the same folder. A second person can do the same and save `similarity_ratings_rater2.csv`, which adds an agreement check. Ratings from an AI model (a large language model reading the same blind sheet) go in `similarity_ratings_llm.csv` and are always reported as an **AI rater**, not as human judgement.

**Step 3 (automatic).** Run this notebook again: it compares the ratings of the three kinds of pair and relates the ratings to the cosine similarity.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 19 — Step 1: the rating sheet (written once, never overwritten)
# ═══════════════════════════════════════════════════════════════════════════
SHEET = R_TABLES / "similarity_rating_sheet.csv"
KEY = R_TABLES / "similarity_rating_key.csv"
if SHEET.exists() and KEY.exists():
    sheet, key = pd.read_csv(SHEET), pd.read_csv(KEY)
    print(f"Rating sheet already exists ({len(sheet)} pairs) — kept unchanged so earlier ratings stay valid.")
else:
    vrng = np.random.default_rng(CFG["validation_seed"])
    n_per = CFG["validation_pairs_per_group"]
    txt = (pd.read_csv(FILES["masked_text"], usecols=["tmdb_id", "overview_masked"]).drop_duplicates("tmdb_id")
             .merge(df[["tmdb_id", "node_id", "primary_genre_model", "vote_count"]], on="tmdb_id"))
    txt = txt[txt["vote_count"] >= CFG["validation_min_votes"]].set_index("node_id")
    ed = pd.read_csv(NETWORK_DIR / "semantic_network_edges_mutual_knn.csv", usecols=["source_node_id", "target_node_id"])
    ed = ed[ed["source_node_id"].isin(txt.index) & ed["target_node_id"].isin(txt.index)]
    linked_set = set(map(tuple, np.sort(ed.values, axis=1)))
    used = set()

    def take(pairs, group):
        out = []
        for a, b in pairs:
            a, b = int(a), int(b)
            if a == b or (min(a, b), max(a, b)) in linked_set and group != "linked" or a in used or b in used:
                continue
            used.update([a, b])
            out.append((a, b, group))
            if len(out) == n_per:
                break
        return out

    pairs = take(ed.values[vrng.permutation(len(ed))], "linked")
    ids = txt.index.values
    by_genre = {g: s.index.values for g, s in txt.groupby("primary_genre_model") if len(s) > 1}
    big = [g for g in by_genre if g != "Other"]
    same = []
    while len(same) < 50 * n_per and big:
        g = big[vrng.integers(len(big))]
        a, b = vrng.choice(by_genre[g], 2, replace=False)
        same.append((a, b))
    pairs += take(same, "same genre")
    pairs += take(vrng.choice(ids, size=(50 * n_per, 2)), "random")
    pairs = [pairs[i] for i in vrng.permutation(len(pairs))]

    pos = pd.Series(np.arange(len(emb_index)), index=emb_index["node_id"].values)
    cos = [float(E[pos[a]] @ E[pos[b]]) for a, b, _ in pairs]
    key = pd.DataFrame({"pair_id": np.arange(1, len(pairs) + 1), "group": [g for *_, g in pairs],
                        "node_a": [a for a, *_ in pairs], "node_b": [b for _, b, _ in pairs], "cosine_similarity": cos})
    key["title_a"] = key["node_a"].map(df.set_index("node_id")["title"])
    key["title_b"] = key["node_b"].map(df.set_index("node_id")["title"])
    sheet = pd.DataFrame({"pair_id": key["pair_id"], "overview_a": key["node_a"].map(txt["overview_masked"]),
                          "overview_b": key["node_b"].map(txt["overview_masked"]), "rating": ""})
    sheet.to_csv(SHEET, index=False)
    key.to_csv(KEY, index=False)
    print(f"  ✓ tables/robustness/{SHEET.name} ({len(sheet)} pairs: {key['group'].value_counts().to_dict()})")
    print(f"  ✓ tables/robustness/{KEY.name} (pair types and titles — do not open before rating)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 20 — Step 3: analyse the ratings (runs once a ratings file exists)
# ═══════════════════════════════════════════════════════════════════════════
from scipy import stats
rating_files = sorted(R_TABLES.glob("similarity_ratings_*.csv"))
GROUPS = ["linked", "same genre", "random"]
if not rating_files:
    print("No ratings yet: fill similarity_rating_sheet.csv and save it as similarity_ratings_rater1.csv "
          "(see the instructions above), then run this cell again.")
    NOTES.append("Human similarity check: not yet rated (outputs/tables/robustness/similarity_rating_sheet.csv).")
else:
    rt = []
    for f in rating_files:
        r = pd.read_csv(f)[["pair_id", "rating"]]
        r["rating"] = pd.to_numeric(r["rating"], errors="coerce")
        name = f.stem.replace("similarity_ratings_", "")
        r = r[~r["pair_id"].isin(CFG["unblinded_pairs"].get(name, []))]    # pairs this rater saw in the key
        rt.append(r.dropna().assign(rater=name))
    rt = pd.concat(rt).merge(key[["pair_id", "group", "cosine_similarity"]], on="pair_id")
    per_pair = rt.groupby(["pair_id", "group", "cosine_similarity"], as_index=False)["rating"].mean()
    raters = sorted(rt["rater"].unique())
    ai = [x for x in raters if x in CFG["ai_raters"]]
    who = (f"{len(raters) - len(ai)} human rater(s)" if len(ai) < len(raters) else "") \
          + (" and " if ai and len(ai) < len(raters) else "") + (f"{len(ai)} AI rater (LLM; not a human judgement)" if ai else "")
    print(f"{who}, {len(per_pair)} rated pairs")

    def boot_ci(x, n=2000):
        bs = np.random.default_rng(CFG["random_state"]).choice(x, (n, len(x))).mean(axis=1)
        return np.quantile(bs, [0.025, 0.975])
    summ = []
    for g in GROUPS:
        x = per_pair.loc[per_pair["group"] == g, "rating"].values
        if len(x):
            lo, hi = boot_ci(x)
            summ.append({"group": g, "pairs": len(x), "mean_rating": x.mean(), "ci_low": lo, "ci_high": hi,
                         "share_rated_4_or_5": (x >= 4).mean(), "mean_cosine": per_pair.loc[per_pair["group"] == g, "cosine_similarity"].mean()})
    summ = pd.DataFrame(summ)
    lk, sg, rd = (per_pair.loc[per_pair["group"] == g, "rating"] for g in GROUPS)
    t_ls = stats.mannwhitneyu(lk, sg, alternative="greater")
    t_lr = stats.mannwhitneyu(lk, rd, alternative="greater")
    rho = stats.spearmanr(per_pair["cosine_similarity"], per_pair["rating"])
    tests_v = pd.DataFrame([
        {"test": "linked > same genre (Mann–Whitney, one-sided)", "statistic": t_ls.statistic, "p_value": t_ls.pvalue},
        {"test": "linked > random (Mann–Whitney, one-sided)", "statistic": t_lr.statistic, "p_value": t_lr.pvalue},
        {"test": "rating vs cosine similarity (Spearman ρ)", "statistic": rho.statistic, "p_value": rho.pvalue}])
    if rt["rater"].nunique() >= 2:
        from sklearn.metrics import cohen_kappa_score
        w = rt.pivot_table(index="pair_id", columns="rater", values="rating").dropna()
        r1, r2 = w.columns[:2]
        tests_v.loc[len(tests_v)] = {"test": f"agreement {r1} vs {r2} (quadratic-weighted κ)",
                                     "statistic": cohen_kappa_score(w[r1].astype(int), w[r2].astype(int), weights="quadratic"),
                                     "p_value": np.nan}
    display(summ.round(3)); display(tests_v.round(4))
    summ["raters"] = who
    save_tab(summ.round(4), "similarity_validation_summary.csv")
    save_tab(tests_v.round(6), "similarity_validation_tests.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.2]})
    ax = axes[0]
    gcol = {"linked": SERIES[0], "same genre": SERIES[3], "random": AXIS}
    for i, r in summ.iterrows():
        ax.barh(i, r["mean_rating"], color=gcol[r["group"]], height=0.55)
        ax.hlines(i, r["ci_low"], r["ci_high"], color=INK_2, linewidth=1.5)
        ax.text(r["ci_high"] + 0.05, i, f"{r['mean_rating']:.2f}  ({r['share_rated_4_or_5']:.0%} rated 4–5)", va="center", fontsize=9, color=INK_2)
    ax.set_yticks(range(len(summ)), [f"{g} ({n})" for g, n in zip(summ["group"], summ["pairs"])])
    ax.invert_yaxis()
    ax.set_xlim(1, 5.9)
    ax.set_xlabel("mean rating (1 = different stories, 5 = same kind of story)")
    ax.grid(axis="y", visible=False)
    ax.set_title("Ratings by kind of pair", fontsize=11, pad=8)
    ax = axes[1]
    jit = np.random.default_rng(1).uniform(-0.15, 0.15, len(per_pair))
    for g in GROUPS:
        m = per_pair["group"] == g
        ax.scatter(per_pair.loc[m, "cosine_similarity"], per_pair.loc[m, "rating"] + jit[m.values], s=22,
                   color=gcol[g], alpha=0.85, label=g, edgecolor="none")
    ax.set_xlabel("cosine similarity of the two overviews (MiniLM)")
    ax.set_ylabel("rating")
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.legend(loc="upper left")
    ax.set_title(f"Rating vs embedding similarity (Spearman ρ = {rho.statistic:.2f})", fontsize=11, pad=8)
    fig_header(fig, "Do films with similar overviews tell similar stories?",
               f"Blind ratings of {len(per_pair)} film pairs (titles and pair type hidden) · {who} · bars: mean with 95% bootstrap CI")
    save(fig, per_pair, "fig_05_similarity_validation")

    sm = summ.set_index("group")
    left_out = {k: v for k, v in CFG["unblinded_pairs"].items() if k in raters and v}
    NOTES.append(f"Similarity check ({who}, {len(per_pair)} blind pairs"
                 + (f"; left out because the rater saw them in the key first: " + ", ".join(f"{k} pairs {v}" for k, v in left_out.items()) if left_out else "")
                 + "): mean rating "
                 + ", ".join(f"{g} {sm.loc[g, 'mean_rating']:.2f}" for g in GROUPS if g in sm.index)
                 + f" (1–5). Linked pairs are rated higher than same-genre pairs (p {fp(t_ls.pvalue)}) and random pairs "
                 f"(p {fp(t_lr.pvalue)}); rating vs cosine similarity ρ = {rho.statistic:.2f} (p {fp(rho.pvalue)})."
                 + (f" Agreement between raters: weighted κ = {tests_v.iloc[-1]['statistic']:.2f}." if rt['rater'].nunique() >= 2 else ""))
    print(NOTES[-1])

## Part 6 — Story vs stars: does plot position matter more for films without a star?

Parts 1–4 show that a film's place in plot space adds little to the usual explanations *on average*. One reason could be that audiences and distributors judge most films by their cast first. **Theory:** the story should carry the most weight when there is no star to sell the film, so plot position should matter more for films without a star.

The test is fixed before looking at the result and run once:
* the full Notebook 05 model (M4: year, genre, runtime, English, plot position, star power, community), unchanged;
* plus **each plot measure × has-a-star** for four measures: betweenness, participation, clustering and novelty;
* for the two outcomes of the main text, **revenue percentile** and **rating**.

That gives 8 interaction tests, corrected together with Benjamini–Hochberg as their own family. The main 36 tests of Part 1 are left exactly as they are. Whatever the result, it is reported. A null result here means plot position matters about equally little with or without a star.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 21 — Part 6: plot position × has-a-star (fixed in advance: 4 measures × 2 outcomes)
# ═══════════════════════════════════════════════════════════════════════════
MOD_TERMS = ["z_log_betweenness", "z_participation", "z_clustering", "z_novelty"]
MOD_OUT = ["revenue_percentile_year", "vote_average"]
inter = " + ".join(f"{t}:has_star" for t in MOD_TERMS)
mod_rows, mod_fit = [], []
for y in MOD_OUT:
    d = df.dropna(subset=[y] + NB05_VARS + NET_TERMS)
    m4 = fit(f"{y} ~ {CTRL} + {' + '.join(NET_TERMS)} + {STAR} + {COMM}", d)
    mi = fit(f"{y} ~ {CTRL} + {' + '.join(NET_TERMS)} + {STAR} + {COMM} + {inter}", d)
    joint = mi.wald_test(" , ".join(f"{t}:has_star = 0" for t in MOD_TERMS), scalar=True)
    mod_fit.append({"outcome": y, "n_obs": int(mi.nobs), "films_with_star": int(d["has_star"].sum()),
                    "r2_M4": m4.rsquared, "r2_with_interactions": mi.rsquared, "joint_interaction_p": float(joint.pvalue)})
    for t in MOD_TERMS:
        no_star = mi.t_test(f"{t} = 0")
        with_star = mi.t_test(f"{t} + {t}:has_star = 0")
        diff = mi.t_test(f"{t}:has_star = 0")
        for grp, tt in [("no star", no_star), ("with star", with_star), ("difference", diff)]:
            lo, hi = np.asarray(tt.conf_int()).ravel()
            mod_rows.append({"outcome": y, "term": t, "group": grp, "coef": float(np.asarray(tt.effect).ravel()[0]),
                             "ci_low": lo, "ci_high": hi, "p_value": float(np.asarray(tt.pvalue).ravel()[0]),
                             "n_obs": int(mi.nobs)})
mod = pd.DataFrame(mod_rows)
mod_fit = pd.DataFrame(mod_fit)
dif = mod["group"] == "difference"
mod.loc[dif, "q_value"] = bh(mod.loc[dif, "p_value"])
mod.loc[dif, "survives_fdr"] = mod.loc[dif, "q_value"] < CFG["fdr_q"]
mod["claim"] = mod["term"].map(TERM_NAMES) + " → " + mod["outcome"].map(OUTCOME_NAMES)
display(mod[dif][["claim", "coef", "p_value", "q_value", "survives_fdr"]].round(4))
display(mod_fit.round(4))
save_tab(mod.round(6), "story_vs_stars_interactions.csv")
save_tab(mod_fit.round(6), "story_vs_stars_model_fit.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 22 — Figure: each plot measure for films without and with a star
# ═══════════════════════════════════════════════════════════════════════════
GCOL = {"no star": SERIES[0], "with star": SERIES[1]}
fig, axes = plt.subplots(1, len(MOD_OUT), figsize=(13, 0.8 * len(MOD_TERMS) + 2.3), sharey=True)
for ax, y in zip(axes, MOD_OUT):
    t = mod[mod["outcome"] == y]
    yy = np.arange(len(MOD_TERMS))[::-1]
    for off, g in [(0.15, "no star"), (-0.15, "with star")]:
        r = t[t["group"] == g].set_index("term").reindex(MOD_TERMS)
        ax.hlines(yy + off, r["ci_low"], r["ci_high"], color=GCOL[g], linewidth=2.5, alpha=0.55)
        ax.scatter(r["coef"], yy + off, color=GCOL[g], s=38, zorder=3, label=g)
    d_ = t[t["group"] == "difference"].set_index("term").reindex(MOD_TERMS)
    for yi, r in zip(yy, d_.itertuples()):
        ax.annotate(f"difference p = {fp(r.p_value)}, q = {fp(r.q_value)}", (1.0, yi), xycoords=("axes fraction", "data"),
                    xytext=(-4, 0), textcoords="offset points", ha="right", va="center", fontsize=7.5, color=INK_2)
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_yticks(yy, [TERM_NAMES[x] for x in MOD_TERMS])
    ax.set_xlabel(f"effect of +1 SD on {OUTCOME_NAMES[y]}")
    ax.grid(axis="y", visible=False)
    f_ = mod_fit.set_index("outcome").loc[y]
    ax.set_title(f"{OUTCOME_NAMES[y]} · {int(f_.n_obs):,} films, {int(f_.films_with_star):,} with a star", fontsize=10.5, pad=6, loc="left")
axes[0].legend(loc="lower left", fontsize=8.5)
fig_header(fig, "Story vs stars: does plot position matter more when there is no star?",
           "Notebook 05 model (M4) plus plot measure × has-a-star · 95% CI clustered by community · "
           "q = Benjamini–Hochberg over the 8 interaction tests")
save(fig, mod, "fig_06_story_vs_stars")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 23 — Notes for Part 6
# ═══════════════════════════════════════════════════════════════════════════
dd = mod[mod["group"] == "difference"]
grp_ = lambda y, t, g: mod[(mod["outcome"] == y) & (mod["term"] == t) & (mod["group"] == g)].iloc[0]
def mod_sentence(r):
    a, b = grp_(r.outcome, r.term, "no star"), grp_(r.outcome, r.term, "with star")
    return (f"{r.claim}: no star {a.coef:+.4f} (p {fp(a.p_value)}), with star {b.coef:+.4f} (p {fp(b.p_value)}), "
            f"difference p {fp(r.p_value)}, q {fp(r.q_value)} — stronger "
            + ("without a star, as the hypothesis predicts" if abs(a.coef) > abs(b.coef) else "with a star, against the hypothesis"))
n_raw_, n_fdr_ = int((dd["p_value"] < 0.05).sum()), int(dd["survives_fdr"].sum())
NOTES.append(f"Story vs stars (plot measure × has-a-star, fixed in advance; {len(dd)} tests, own FDR family): "
             f"{n_raw_} of {len(dd)} interactions have p < 0.05, {n_fdr_} survive the correction. Joint test per outcome: "
             + "; ".join(f"{OUTCOME_NAMES[r.outcome]} p {fp(r.joint_interaction_p)}" for r in mod_fit.itertuples()) + ".")
show_ = dd[dd["p_value"] < 0.05]
if len(show_):
    NOTES.append("  " + "; ".join(mod_sentence(r) for r in show_.itertuples()) + ".")
else:
    NOTES.append("  No plot measure works differently for films with and without a star: plot position matters about "
                 "equally little either way.")
print("\n".join(NOTES[-2:]))

## Part 7 — Few clusters: do the p-values hold with only 36 communities?

All models cluster their standard errors by community, so that films in the same narrative are not counted as independent evidence. There are only **36 clusters**. With that few, the usual cluster-robust p-values can be somewhat too small.

The standard check is the **wild cluster bootstrap** (Cameron, Gelbach & Miller 2008; Webb 2023). The outcome is regenerated many times under "no effect", shaking whole communities at once, and the observed t-statistic is compared with that distribution. It is run here for every result that survives the FDR correction (Parts 1 and 6), on exactly the same models and films. A result holds if its bootstrap p-value stays below 0.05.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 24 — Part 7: wild cluster bootstrap-t for the results that survive FDR
# ═══════════════════════════════════════════════════════════════════════════
WEBB = np.array([-np.sqrt(1.5), -1, -np.sqrt(0.5), np.sqrt(0.5), 1, np.sqrt(1.5)])

def wild_cluster_p(formula, data, term, B=9999, seed=CFG["random_state"]):
    """Restricted wild cluster bootstrap-t (Webb weights) for H0: term = 0, clusters = community_group.
    Uses the same cluster-robust t-statistic as fit(); B draws in blocks of 1,000."""
    m = smf.ols(formula, data=data)
    X, y = m.exog, m.endog
    k = m.exog_names.index(term)
    g = pd.factorize(data.loc[m.data.row_labels, "community_group"])[0]
    G, (n, p) = g.max() + 1, X.shape
    XtXi = np.linalg.pinv(X.T @ X)
    a = XtXi[k] @ X.T                                            # coefficient k = a @ y
    Xr = np.delete(X, k, axis=1)
    yr = Xr @ np.linalg.lstsq(Xr, y, rcond=None)[0]              # fit with the effect set to zero
    ur = y - yr
    H = np.zeros((n, G)); H[np.arange(n), g] = 1                 # film → cluster
    ay, au, A = H.T @ (a * yr), H.T @ (a * ur), H.T @ (a[:, None] * X)
    B0, Bm = XtXi @ (X.T @ yr), XtXi @ (X.T @ (ur[:, None] * H))
    c = G / (G - 1) * (n - 1) / (n - p)                          # same small-sample factor as statsmodels
    def t_boot(w):
        s = (ay - A @ B0)[:, None] + au[:, None] * w - A @ (Bm @ w)
        return (a @ yr + au @ w) / np.sqrt(c * (s ** 2).sum(axis=0))
    e = y - X @ (XtXi @ (X.T @ y))
    t_obs = (a @ y) / np.sqrt(c * ((H.T @ (a * e)) ** 2).sum())
    rng = np.random.default_rng(seed)
    ts = np.concatenate([t_boot(WEBB[rng.integers(0, 6, size=(G, 1000))]) for _ in range(-(-B // 1000))])[:B]
    return float(t_obs), float((np.abs(ts) >= abs(t_obs)).mean()), int(G), int(n)

T3 = "crowding_z + I(crowding_z**2) + foresight_z + competition_z"
NEED6 = NB05_VARS + ["crowding_z", "foresight_z", "competition_z"]
boot_rows = []
for r in tests[tests["survives_fdr"]].itertuples():
    if r.source.startswith("Notebook 05"):
        d = df.dropna(subset=[r.outcome] + NB05_VARS + NET_TERMS)
        f = f"{r.outcome} ~ {CTRL} + {' + '.join(NET_TERMS)} + {STAR} + {COMM}"
    else:
        d = df.dropna(subset=[r.outcome] + NEED6)
        f = f"{r.outcome} ~ {BASE} + {T3}"
    boot_rows.append({"claim": r.claim, "source": r.source, "outcome": r.outcome, "term": r.term, "coef": r.coef,
                      "p_cluster": r.p_value, "q_value": r.q_value, "formula": f, "data": d})
for r in mod[(mod["group"] == "difference") & (mod["survives_fdr"] == True)].itertuples():
    d = df.dropna(subset=[r.outcome] + NB05_VARS + NET_TERMS)
    f = f"{r.outcome} ~ {CTRL} + {' + '.join(NET_TERMS)} + {STAR} + {COMM} + {inter}"
    boot_rows.append({"claim": f"{TERM_NAMES[r.term]} × has-a-star → {OUTCOME_NAMES[r.outcome]}", "source": "Part 6",
                      "outcome": r.outcome, "term": f"{r.term}:has_star", "coef": r.coef, "p_cluster": r.p_value,
                      "q_value": r.q_value, "formula": f, "data": d})
for b in boot_rows:
    b["t_cluster"], b["p_wild_bootstrap"], b["clusters"], b["n_obs"] = wild_cluster_p(b.pop("formula"), b.pop("data"), b["term"])
    b["holds"] = b["p_wild_bootstrap"] < 0.05
boot = pd.DataFrame(boot_rows)
display(boot[["claim", "coef", "t_cluster", "p_cluster", "p_wild_bootstrap", "holds"]].round(4))
save_tab(boot.round(6), "wild_cluster_bootstrap.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 25 — Notes for Part 7
# ═══════════════════════════════════════════════════════════════════════════
NOTES.append(f"Few clusters ({boot['clusters'].iloc[0]} communities; wild cluster bootstrap-t, Webb weights, 9,999 draws): "
             f"{int(boot['holds'].sum())} of {len(boot)} results that survive the FDR correction keep p < 0.05. "
             + "; ".join(f"{r.claim}: p {fp(r.p_cluster)} (clustered) → {fp(r.p_wild_bootstrap)} (bootstrap)"
                         for r in boot.itertuples()) + ".")
print(NOTES[-1])

## Notes for the thesis

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15 — Write the notes file
# ═══════════════════════════════════════════════════════════════════════════
text = "\n".join(["Robustness notes (Notebook 07)", ""] + NOTES +
                 ["", "Communities in rebuilt networks come from the best of several Louvain runs, not the consensus of "
                  "Notebook 03; the community control in all models is the 35 labelled communities of the main network. "
                  "All results are associations, not causal effects."])
(R_TABLES / "robustness_notes.txt").write_text(text, encoding="utf-8")
print(text)
print("\n  ✓ tables/robustness/robustness_notes.txt")